In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test_sparkdataframe') \
    .getOrCreate()

26/03/02 20:03:10 WARN Utils: Your hostname, iseohuiui-MacBookAir.local resolves to a loopback address: 127.0.0.1; using 192.168.219.109 instead (on interface en0)
26/03/02 20:03:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 20:03:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/02 20:03:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/02 20:03:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/02 20:03:10 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
df_green = spark.read.parquet('data/pq/green/*/*')

In [3]:
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [4]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')

In [5]:
df_yellow = df_yellow \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [6]:
common_colums = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_colums.append(col)

In [7]:
from pyspark.sql import functions as F

In [8]:
df_green_sel = df_green \
    .select(common_colums) \
    .withColumn('service_type', F.lit('green'))

In [9]:
df_yellow_sel = df_yellow \
    .select(common_colums) \
    .withColumn('service_type', F.lit('yellow'))

In [10]:
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [11]:
df_trips_data.groupBy("service_type").count().show()

+------------+--------+
|service_type|   count|
+------------+--------+
|       green|  570466|
|      yellow|15000700|
+------------+--------+



In [12]:
df_trips_data

DataFrame[VendorID: int, pickup_datetime: timestamp, dropoff_datetime: timestamp, store_and_fwd_flag: string, RatecodeID: int, PULocationID: int, DOLocationID: int, passenger_count: int, trip_distance: double, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, ehail_fee: double, improvement_surcharge: double, total_amount: double, payment_type: int, trip_type: int, congestion_surcharge: double, service_type: string]

In [13]:
df_result = df_trips_data\
.withColumnRenamed("PULocationID", "revenue_zone")\
.withColumn("revenue_month", date_trunc('month', "pickup_datetime"))\
.groupBy("revenue_zone", "revenue_month", "service_type")\
.agg(
    sum("fare_amount").alias("revenue_monthly_fare"),
    sum("extra").alias("revenue_monthly_extra"),
    sum("mta_tax").alias("revenue_monthly_mta_tax"),
    sum("tip_amount").alias("revenue_monthly_tip_amount"),
    sum("tolls_amount").alias("revenue_monthly_tolls_amount"),
    sum("improvement_surcharge").alias("revenue_monthly_improvement_surcharge"),
    sum("total_amount").alias("revenue_monthly_total_amount"),
    sum("congestion_surcharge").alias("revenue_monthly_congestion_surcharge"),
    avg("passenger_count").alias("avg_monthly_passenger_count"),
    avg("trip_distance").alias("avg_monthly_trip_distance")
)

In [14]:
df_result.show()

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_passenger_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|         242

In [15]:
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')

In [16]:
spark